# Set Up Colab Workspace

In [ ]:
# Install dependencies
!pip install transformers evaluate accelerate huggingface_hub jsonlines
!pip install git+https://github.com/JetBrains-Research/mxeval.git
!pip install -U datasets

# Download and set up Kotlin compiler
!rm -rf kotlin-compiler-1.9.24.zip
!wget https://github.com/JetBrains/kotlin/releases/download/v1.9.24/kotlin-compiler-1.9.24.zip
!unzip -o kotlin-compiler-1.9.24.zip

import os
os.environ["PATH"] += ":/content/kotlinc/bin"

In [ ]:
# Download src code
!rm -rf LoResLift src
!git clone https://github.com/Dang-Hoang-Tung/LoResLift.git
!mv LoResLift/src .
!rm -rf LoResLift

In [ ]:
# Connect to Drive to save outputs
from google.colab import drive
drive.mount("/content/drive")

# Run Config

In [ ]:
HF_TOKEN = "<YOUR HUGGINGFACE TOKEN>"

# --- Control variables (edit these) ---
PIPELINE_TYPE = "DIRECT"  # DIRECT | PIVOT_LLM | PIVOT_RB | PIVOT_RB_LLM
MODEL_ID = "microsoft/wavecoder-ultra-6.7b"
DRIVE_PROJECT_DIR = "drive/MyDrive/Colab Notebooks/LoResLift Project/HumanEval/"

KOTLIN_DATA_PATH = "src/HumanEval/data/HumanEval_kotlin_v1.1.jsonl"
JAVA_DATA_PATH   = "src/HumanEval/data/HumanEval_java_v1.1.jsonl"

import os
import time

from src.HumanEval.config import make_paths

# --- Derived paths ---
paths = make_paths(MODEL_ID, PIPELINE_TYPE, DRIVE_PROJECT_DIR)

notebook_start = time.time()
print(f"Pipeline: {PIPELINE_TYPE}  |  Model: {MODEL_ID}")

# --- Mount Drive and create output dirs ---
os.makedirs(paths.output_dir, exist_ok=True)
os.makedirs(paths.output_pivot_dir, exist_ok=True)

# Pre-run Set Up

## Load dataset

In [ ]:
from src.HumanEval.dataset import load_kotlin_dataset, load_java_dataset

problem_list, problem_dict, kotlin_prompt_dict, kotlin_signatures = load_kotlin_dataset(KOTLIN_DATA_PATH)
java_prompt_dict = load_java_dataset(JAVA_DATA_PATH)

print(f"Loaded {len(problem_dict)} Kotlin problems")

## Load model

In [ ]:
from src.HumanEval.model import load_model

tokenizer, model = load_model(MODEL_ID, HF_TOKEN)

# Run Pipeline

## Direct Flows

### Run Direct LR Flow

In [ ]:
from src.HumanEval.pipelines import run_direct

if PIPELINE_TYPE == "DIRECT":
    run_direct(kotlin_prompt_dict, model, tokenizer, paths.output_eval_file)

### Run Direct HR Flow

In [ ]:
# TODO: add direct HR pipeline

## Pivot Flows (LoResLift)

### Pivot Set Up

In [ ]:
from src.HumanEval.pipelines import load_failed_samples

if PIPELINE_TYPE != "DIRECT":
    results_direct_path = paths.output_group_dir + "/" + "results-direct/output_eval.jsonl_results.jsonl"
    failed_problem_dict, failed_kotlin_samples = load_failed_samples(results_direct_path, problem_dict)
    print(f"Retrying {len(failed_problem_dict)} failed problems")

### Run Pivot Pipeline

In [ ]:
from src.HumanEval.pipelines import run_pivot_java_gen

if PIPELINE_TYPE != "DIRECT":
    run_pivot_java_gen(java_prompt_dict, failed_problem_dict, model, tokenizer, paths.output_java_file)

### Downward
Translate Java solution back to Kotlin

In [ ]:
from src.HumanEval.pipelines import run_pivot_llm, export_for_rb_translation, run_pivot_rb_cleanup, run_pivot_rb_llm_fixup

# Translate using LLMs
if PIPELINE_TYPE == "PIVOT_LLM":
    run_pivot_llm(
        failed_kotlin_samples, paths.output_java_file, kotlin_signatures,
        tokenizer, model, paths.output_dir, paths.output_eval_file,
    )
    
# Export files, translate with IntelliJ, then continue
if PIPELINE_TYPE in ("PIVOT_RB", "PIVOT_RB_LLM"):
    export_for_rb_translation(paths.output_java_file, paths.output_pivot_dir)
    print("Download the 'to_translate' directory, translate with IntelliJ's built-in converter,")
    print("then upload the .kt files to an adjacent 'translated/' directory.")
    input("Press Enter when finished...")
    
    run_pivot_rb_cleanup(
        paths.output_pivot_dir, paths.output_dir, PIPELINE_TYPE,
        paths.output_file, paths.output_eval_file,
    )

    # Optionally run a final LLM fixup step to try to fix any remaining issues after RB translation
    if PIPELINE_TYPE == "PIVOT_RB_LLM":
        run_pivot_rb_llm_fixup(
            failed_kotlin_samples, kotlin_signatures, paths.output_dir,
            tokenizer, model, paths.output_eval_file,
        )

## Evaluate

In [ ]:
from src.HumanEval.pipelines import run_evaluate

run_evaluate(paths.output_eval_file, problem_dict, paths.results_file)

elapsed = int(time.time() - notebook_start)
hours, remainder = divmod(elapsed, 3600)
minutes, seconds = divmod(remainder, 60)
print(f"Total runtime: {hours:02d}:{minutes:02d}:{seconds:02d}")